# Fine-Tuning AI Search for E-commerce — Workshop Lab

**Welcome.** Over the next 60 minutes you'll go from a stock dense embedding model to a fine-tuned SPLADE + hybrid pipeline, on real Amazon ESCI data (labeled product-search queries with Exact/Substitute/Complement/Irrelevant grades), and watch the numbers move.

### What you'll do

1. **CP1 — Baseline dense (12 min):** run ~10 demo queries against `all-MiniLM-L6-v2` and *feel* the failure modes.
2. **CP2 — Fine-tuned SPLADE (20 min):** see what fine-tuned SPLADE buys you over baseline dense. First nDCG@10 / Recall@10 numbers on the demo set across BM25 + baseline dense + fine-tuned SPLADE.
3. **CP3 — Hybrid fusion, two recipes (13 min):** fuse dense + BM25 *and* dense + fine-tuned SPLADE with RRF using Qdrant's Query API. Same call structure, two different sparse legs.
4. **Wrap (15 min):** full 2K-query eval with 95% CIs across all five approaches, the bad queries / query-routing honesty moment, takeaways.

> **Heads-up:** the indexed catalog is a ~35K-40K-product ESCI subset — *not* the full Amazon catalog. Queries about products outside it return the closest match, not nothing.
>
> **Workshop-specific corpus:** we choose this subset from ESCI relevance labels so every graded product is reachable during evaluation. In production, you would index the full product catalog first, then evaluate against query logs, judgments, clicks, or other relevance data.

## Setup

One cell: pre-flight check, load the ESCI test split from HuggingFace, attach qrels to the demo queries, and instantiate the encoders we'll reuse below.

In [ ]:
from pathlib import Path
import sys, json, uuid, platform
from functools import lru_cache

if sys.version_info >= (3, 14):
    raise RuntimeError(
        f"Python {platform.python_version()} is not supported for this lab. "
        "Use Python 3.12; the current FastEmbed/ONNX Runtime stack can "
        "segfault during local embedding on Python 3.14."
    )

sys.path.insert(0, ".")  # repo root = IDE workspace root
DATA = Path("data")

import pandas as pd
from datasets import load_dataset
from IPython.display import display, Markdown
from qdrant_client import QdrantClient, models
from tqdm.auto import tqdm

from eval import SpladeEncoder
from eval.metrics import ndcg_at_k, recall_at_k, bootstrap_ci, explain_metric
from eval.viewer import compare_results, inspect_sparse_vector, render_query_block
from scripts.setup_collections import (
    COLLECTION,
    DENSE_VECTOR_NAME, BM25_VECTOR_NAME, SPLADE_VECTOR_NAME,
    BM25_MODEL as BM25_MODEL_ID,
    DENSE_MODEL as DENSE_MODEL_ID,
    SPLADE_FINETUNED_MODEL_DEFAULT,
)

# Qdrant + SPLADE encoder
client = QdrantClient(host="localhost", port=6333)
splade_encoder = SpladeEncoder(SPLADE_FINETUNED_MODEL_DEFAULT, device="cpu")

# Confirm the Qdrant collection matches the eval subset we're about to query.
manifest_path = DATA / "corpus_manifest.json"
assert manifest_path.exists(), f"{manifest_path} not found. Run scripts/setup_collections.py first."
manifest = json.loads(manifest_path.read_text())


def _stable_point_id(product_id: str) -> str:
    """Mirror scripts/setup_collections._stable_point_id so we can spot-check."""
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"esci:{product_id}"))


points_count = client.get_collection(COLLECTION).points_count or 0
expected_count = manifest["corpus_product_count"]
assert points_count == expected_count, (
    f"Qdrant collection has {points_count:,} points but manifest expects "
    f"{expected_count:,}. The collection is stale or was provisioned from a "
    f"different ESCI selection. Re-run scripts/setup_collections.py --recreate."
)

sample_pids = manifest["corpus_sample_product_ids"]
sample_uuids = [_stable_point_id(pid) for pid in sample_pids]
found = client.retrieve(collection_name=COLLECTION, ids=sample_uuids, with_payload=False)
assert len(found) == len(sample_uuids), (
    f"Corpus reachability check failed: {len(sample_uuids) - len(found)} of "
    f"{len(sample_uuids)} sample product_ids from the manifest are not in the "
    f"collection. Re-run scripts/setup_collections.py --recreate."
)

splade_encoder.encode(["ping"])  # warm + sanity-check
print(f"[OK] Qdrant up at localhost:6333, products collection has {points_count:,} points (matches manifest)")
print(f"[OK] reachability spot-check: {len(found)}/{len(sample_uuids)} sample products present")

# ---- Demo + bad query bundles ------------------------------------------------
demo_queries = json.loads((DATA / "demo_queries.json").read_text())
bad_queries  = json.loads((DATA / "bad_queries.json").read_text())

# Pull the eval queries + their qrels from Amazon ESCI on HuggingFace.
eval_qids = set(manifest["eval_query_ids"])
print(f"Loading {len(eval_qids):,} ESCI eval queries from HuggingFace...")
_ds = load_dataset(manifest["dataset_id"], split=manifest["split"])

def _normalize_grade(label):
    return str(label)[0].upper() if label else "I"

manifest_small_version = manifest.get("small_version")
_by_qid = {}
for _row in _ds:
    if str(_row.get("product_locale", "us")).lower() != str(manifest["locale"]).lower():
        continue
    if manifest_small_version is not None:
        try:
            if int(_row.get("small_version", manifest_small_version)) != int(manifest_small_version):
                continue
        except (TypeError, ValueError):
            pass
    try:
        _qid = int(_row["query_id"])
    except (KeyError, TypeError, ValueError):
        continue
    if _qid not in eval_qids:
        continue
    if _qid not in _by_qid:
        _by_qid[_qid] = {"query_id": _qid, "query": _row["query"], "qrels": {}}
    _by_qid[_qid]["qrels"][_row["product_id"]] = _normalize_grade(_row.get("esci_label"))

missing = eval_qids - set(_by_qid.keys())
assert not missing, (
    f"{len(missing)} eval query_ids from the manifest were not found in the live "
    f"ESCI split. The dataset mirror likely shifted between provisioning and now."
)
full_2k = [_by_qid[qid] for qid in sorted(eval_qids)]
print(f"[OK] ESCI eval set: {len(full_2k):,} queries loaded")

# Attach qrels to demo queries (their esci_qid is the ESCI query_id and is
# guaranteed to be in the manifest by construction).
for h in demo_queries:
    h["qrels"] = _by_qid[int(h["esci_qid"])]["qrels"]

# SPLADE query vectors are encoded inside search_splade/search_hybrid_splade.
# Those helpers memoize repeated query strings so CP2, CP3, and the wrap do
# not encode the same query twice.

display(Markdown(
    f"**Ready.** Corpus: {points_count:,} products · "
    f"{len(demo_queries)} demo queries · {len(bad_queries)} bad queries · "
    f"{len(full_2k):,} eval queries."
))


def retrieved_ids(points):
    """Project Qdrant points down to their ESCI product_id (or UUID fallback)."""
    return [p.payload.get("product_id", p.id) for p in points]


---
### Build a mini-collection from scratch (2 min)

Before we start querying the prebuilt ~35K-40K-product `products` collection, here's how it was built. Same API surface, scaled down to six products so the whole **create → upsert → query** lifecycle runs in seconds.

`QdrantClient(":memory:")` spins up an embedded Qdrant **inside this kernel** — no second server, no network. The schema we register below — one dense + two sparse named vectors — is exactly what `scripts/setup_collections.py` registers at full workshop-corpus scale.

In [ ]:
# Embedded Qdrant inside the kernel -- no second server, no network.
mini_client = QdrantClient(":memory:")

mini_products = [
    {"product_id": "demo-1", "product_title": "Sony 65 inch 4K HDR Smart TV"},
    {"product_id": "demo-2", "product_title": "LG 55 inch OLED Smart TV"},
    {"product_id": "demo-3", "product_title": "Apple iPhone 13 silicone case, Midnight"},
    {"product_id": "demo-4", "product_title": "Apple iPhone 11 leather case, Saddle Brown"},
    {"product_id": "demo-5", "product_title": "10 gallon glass fish aquarium tank"},
    {"product_id": "demo-6", "product_title": "5 gallon plastic storage bucket"},
]

# Same schema as the real `products` collection: one dense vector + two
# sparse vectors. BM25 uses Modifier.IDF so Qdrant applies inverse-document-
# frequency at query time. SPLADE uses default params because the model
# already bakes term importance into its learned weights -- nothing for
# Qdrant to fix up at query time.
mini_client.create_collection(
    collection_name="mini_products",
    vectors_config={
        DENSE_VECTOR_NAME: models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        BM25_VECTOR_NAME:   models.SparseVectorParams(modifier=models.Modifier.IDF),
        SPLADE_VECTOR_NAME: models.SparseVectorParams(),
    },
)

# Encode SPLADE for each product title up front (one batched forward pass).
# Dense + BM25 use models.Document so qdrant-client resolves them via
# FastEmbed at upsert time. SPLADE encodes explicitly because FastEmbed
# doesn't wrap arbitrary HF SPLADE checkpoints.
mini_splade_vecs = splade_encoder.encode([p["product_title"] for p in mini_products])

points = []
for i, (prod, (sidx, svals)) in enumerate(zip(mini_products, mini_splade_vecs)):
    points.append(models.PointStruct(
        id=i,
        vector={
            DENSE_VECTOR_NAME:  models.Document(text=prod["product_title"], model=DENSE_MODEL_ID),
            BM25_VECTOR_NAME:   models.Document(text=prod["product_title"], model=BM25_MODEL_ID),
            SPLADE_VECTOR_NAME: models.SparseVector(
                indices=list(map(int, sidx)), values=list(map(float, svals)),
            ),
        },
        payload=prod,
    ))

mini_client.upsert(collection_name="mini_products", points=points)

# One query against the dense leg to confirm the schema works end to end.
# This is the same query_points call shape you'll use against the real
# `products` collection for the rest of the lab.
mini_q = "65 inch tv"
mini_results = mini_client.query_points(
    collection_name="mini_products",
    query=models.Document(text=mini_q, model=DENSE_MODEL_ID),
    using=DENSE_VECTOR_NAME, limit=3, with_payload=True,
).points
print(f"query: {mini_q!r}")
for h in mini_results:
    print(f"  {h.score:.3f}  {h.payload['product_title']}")


---
### Shared retrievers

Five retrieval functions — one set, used by every checkpoint below. Each takes a query text, encodes it in the appropriate format, and calls Qdrant.

In [ ]:
def search_dense(query, k=10):
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=query, model=DENSE_MODEL_ID),
        using=DENSE_VECTOR_NAME, limit=k, with_payload=True,
    ).points


def search_bm25(query, k=10):
    return client.query_points(
        collection_name=COLLECTION,
        query=models.Document(text=query, model=BM25_MODEL_ID),
        using=BM25_VECTOR_NAME, limit=k, with_payload=True,
    ).points


@lru_cache(maxsize=4096)
def _splade_vec(query):
    """Encode one query with SPLADE and memoize repeated calls in this kernel."""
    idx, vals = splade_encoder.encode([query])[0]
    return models.SparseVector(
        indices=list(map(int, idx)), values=list(map(float, vals)),
    )


def search_splade(query, k=10):
    return client.query_points(
        collection_name=COLLECTION,
        query=_splade_vec(query),
        using=SPLADE_VECTOR_NAME, limit=k, with_payload=True,
    ).points


def search_hybrid_bm25(query, k=10, candidates=50):
    """Server-side RRF hybrid: dense + BM25. The classic production recipe."""
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=models.Document(text=query, model=DENSE_MODEL_ID),
                            using=DENSE_VECTOR_NAME, limit=candidates),
            models.Prefetch(query=models.Document(text=query, model=BM25_MODEL_ID),
                            using=BM25_VECTOR_NAME, limit=candidates),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=k, with_payload=True,
    ).points


def search_hybrid_splade(query, k=10, candidates=50):
    """Server-side RRF hybrid: dense + fine-tuned SPLADE. The workshop's pitch."""
    return client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=models.Document(text=query, model=DENSE_MODEL_ID),
                            using=DENSE_VECTOR_NAME, limit=candidates),
            models.Prefetch(query=_splade_vec(query),
                            using=SPLADE_VECTOR_NAME, limit=candidates),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=k, with_payload=True,
    ).points


RETRIEVERS = {
    "BM25":              search_bm25,
    "Dense":             search_dense,
    "SPLADE":            search_splade,
    "Hybrid (D+BM25)":   search_hybrid_bm25,
    "Hybrid (D+SPLADE)": search_hybrid_splade,
}


---
## CP1 — Baseline dense (qualitative, 12 min)

**Goal:** see what `all-MiniLM-L6-v2` does on real e-commerce queries before we touch anything else.

**No aggregate metric yet.** Just look at the top-10 results. Each row is tagged with its ESCI grade (E/S/C/I) — watch for cases where the baseline returns a **Substitute** at rank 1 with the **Exact** buried deeper. The `65 lg tv` query (the same one from the hook slide) is one of the cleanest examples.

We'll label the failure modes we spot after the queries run.

In [ ]:
# Run baseline dense across all 10 demo queries; cache for CP2 side-by-side.
cp1_results = {q["id"]: search_dense(q["text"]) for q in demo_queries}

# Show three focus demo queries in full so the room *feels* the failure modes,
# then summarize the other seven in a compact table.
#   - specificity (size+brand)  -> ele_65_lg_tv (matches the slide hook)
#   - identity (brand+model)    -> ele_apple_iphone_11_case
#   - specificity (quantity)    -> hom_11_piece_knife_set_without_block
focus_ids = [
    "ele_65_lg_tv",
    "ele_apple_iphone_11_case",
    "hom_11_piece_knife_set_without_block",
]

for qid in focus_ids:
    q = next(h for h in demo_queries if h["id"] == qid)
    display(Markdown(f"### `{q['id']}` — *{q['category']}* — **{q['text']}**"))
    compare_results(
        query=q["text"],
        models_results={"baseline_dense": cp1_results[qid]},
        esci_qrels=q["qrels"],
    )

# Compact summary for the other seven demo queries.
_rows = []
for h in demo_queries:
    if h["id"] in focus_ids:
        continue
    pts = cp1_results[h["id"]]
    top = pts[0] if pts else None
    top_pid = top.payload.get("product_id", top.id) if top else None
    top_title = top.payload.get("product_title", top_pid) if top else "—"
    grade = h["qrels"].get(top_pid, "—") if top_pid else "—"
    _rows.append({
        "query_id": h["id"],
        "query_text": h["text"],
        "rank-1 title": str(top_title),
        "rank-1 grade": grade,
    })
display(Markdown("#### Remaining seven demo queries — rank-1 snapshot"))
display(pd.DataFrame(_rows).style.hide(axis="index"))


### Label what you see

Take 2 minutes. For the three focus queries above, jot down which failure mode(s) jumped out:

- **Substitute-at-top** — wrong variant ("65 inch" query → 55 inch result; "11 piece" → 7 piece)
- **Complement-leak** — accessory instead of the product (iPhone query → iPhone case)
- **Missed-attribute** — color / size / brand / quantity ignored
- **Weak-exact-match** — near-paraphrase, not the exact SKU

| Query | Failure modes that jumped out |
|---|---|
| `ele_65_lg_tv` |  |
| `ele_apple_iphone_11_case` |  |
| `hom_11_piece_knife_set_without_block` |  |

We'll share out as a group before moving to CP2.

In [ ]:
# Scratch -- your notes / quick experiments


---
## CP2 — Fine-tuned SPLADE (first numbers, 20 min)

**Goal:** see what a SPLADE model fine-tuned on Amazon ESCI buys you over the dense baseline.

Same demo queries. Two retrievers side-by-side (baseline dense + fine-tuned SPLADE) on the focus demo queries. Then the first aggregate numbers across the demo set — illustrative only; the headline 2K claim lands in the wrap.

In [ ]:
splade_results = {q["id"]: search_splade(q["text"]) for q in demo_queries}

# Side-by-side on two demo queries where the lift is most legible.
cp2_focus = ["ele_65_lg_tv", "hom_11_piece_knife_set_without_block"]
for qid in cp2_focus:
    q = next(h for h in demo_queries if h["id"] == qid)
    display(Markdown(f"### `{q['id']}` — **{q['text']}**"))
    compare_results(
        query=q["text"],
        models_results={
            "baseline_dense": cp1_results[qid],
            "splade_finetuned": splade_results[qid],
        },
        esci_qrels=q["qrels"],
    )

# First metric reveal: demo-mean nDCG@10 / Recall@10 across BM25 + Dense + SPLADE.
# Illustrative only -- n=10 is too small for stable numbers. Headline 2K claim in wrap.
def demo_metrics(retrievers):
    per_q = {name: {"ndcg": [], "recall": []} for name in retrievers}
    for h in demo_queries:
        for name, fn in retrievers.items():
            ids = retrieved_ids(fn(h["text"]))
            per_q[name]["ndcg"].append(ndcg_at_k(ids, h["qrels"], k=10))
            per_q[name]["recall"].append(recall_at_k(ids, h["qrels"], k=10))
    return per_q

per_q_3way = demo_metrics({
    "BM25":   search_bm25,
    "Dense":  search_dense,
    "SPLADE": search_splade,
})

_rows = []
for name, scores in per_q_3way.items():
    _rows.append({
        "approach": name,
        "nDCG@10":  sum(scores["ndcg"])  / len(scores["ndcg"]),
        "Recall@10": sum(scores["recall"]) / len(scores["recall"]),
    })
_df = pd.DataFrame(_rows)
_df["nDCG@10 (plain English)"]   = _df.apply(lambda r: explain_metric("nDCG@10",   r["nDCG@10"]),   axis=1)
_df["Recall@10 (plain English)"] = _df.apply(lambda r: explain_metric("Recall@10", r["Recall@10"]), axis=1)
display(Markdown("#### Demo-set metrics (n=10) — *illustrative, not the headline claim*"))
display(_df.style.hide(axis="index").format({"nDCG@10": "{:.3f}", "Recall@10": "{:.3f}"}))


### Peek inside a sparse vector (2 min)

Sparse is **interpretable** in a way dense isn't. Below we encode the query `"10 gallon fish tank"` with fine-tuned SPLADE and look at the top-weighted tokens. You should see the obvious literals — `tank`, `gallon`, `fish`, `10` — but also notice the **learned term expansions**: tokens like `tanks`, `capacity`, and `aquarium` that weren't typed but appear because fine-tuning taught the model they're semantically related to fish-tank product titles. That's the lift over raw lexical matching.

*(The viewer filters punctuation/stopwords by default; pass `skip_uninformative=False` to see the raw top-K.)*

In [ ]:
vocab = json.loads((DATA / "splade_vocab.json").read_text())
vocab = {int(k): v for k, v in vocab.items()}  # JSON keys come in as strings

q_idx, q_vals = splade_encoder.encode(["10 gallon fish tank"])[0]
inspect_sparse_vector((q_idx, q_vals), vocab=vocab, top_k=12)


---
## CP3 — Hybrid fusion, two recipes (13 min)

**Goal:** combine dense with a sparse leg using **Reciprocal Rank Fusion** in a single Qdrant Query API call, and compare two recipes side by side:

- **Hybrid (D+BM25)** — dense + BM25. The classic production recipe.
- **Hybrid (D+SPLADE)** — dense + fine-tuned SPLADE. The workshop's pitch.

Same RRF call structure, two different sparse legs. Does fusing dense with BM25 match fusing dense with fine-tuned SPLADE?

In [ ]:
hybrid_bm25_results   = {q["id"]: search_hybrid_bm25(q["text"])   for q in demo_queries}
hybrid_splade_results = {q["id"]: search_hybrid_splade(q["text"]) for q in demo_queries}

# Continue the slide-hook through-line: 65 lg tv lands here so attendees see
# the same query from slide 2 -> CP1 -> CP2 -> CP3.
qid = "ele_65_lg_tv"
q = next(h for h in demo_queries if h["id"] == qid)
display(Markdown(f"### Two hybrid recipes on `{qid}` — **{q['text']}**"))
compare_results(
    query=q["text"],
    models_results={
        "splade_finetuned": splade_results[qid],
        "hybrid_bm25":   hybrid_bm25_results[qid],
        "hybrid_splade": hybrid_splade_results[qid],
    },
    esci_qrels=q["qrels"],
)

# 5-way demo-set refresh -- same caveat as CP2 (illustrative, not the headline).
per_q_5way = demo_metrics(RETRIEVERS)

_rows = []
for name in ("BM25", "Dense", "SPLADE", "Hybrid (D+BM25)", "Hybrid (D+SPLADE)"):
    _rows.append({
        "approach": name,
        "nDCG@10":  sum(per_q_5way[name]["ndcg"])  / len(per_q_5way[name]["ndcg"]),
        "Recall@10": sum(per_q_5way[name]["recall"]) / len(per_q_5way[name]["recall"]),
    })
_df = pd.DataFrame(_rows)
_df["nDCG@10 (plain English)"]   = _df.apply(lambda r: explain_metric("nDCG@10",   r["nDCG@10"]),   axis=1)
_df["Recall@10 (plain English)"] = _df.apply(lambda r: explain_metric("Recall@10", r["Recall@10"]), axis=1)
display(Markdown("#### Demo-set 5-way metrics (n=10) — *illustrative*"))
display(_df.style.hide(axis="index").format({"nDCG@10": "{:.3f}", "Recall@10": "{:.3f}"}))


---
## Wrap — the headline numbers, where it breaks, and your turn (15 min)

### The authoritative claim — full 2K eval with 95% CIs

Computed **live** against the full 2K ESCI test split. **This is the number we stand behind.** Bootstrap 95% CIs come from 1,000 resamples per approach.

Running the full 2K eval live takes a few minutes on a 4-vCPU machine (exact range CPU-dependent). SPLADE query vectors are encoded on first use and memoized so the SPLADE and Hybrid (D+SPLADE) rows share the same query vector. Watch the progress bar; the headline table renders underneath when it's done.

In [ ]:
per_query = {name: {"ndcg": [], "recall": []} for name in RETRIEVERS}

for item in tqdm(full_2k, desc="2K eval", unit="q"):
    q_text, q_rels = item["query"], item["qrels"]
    for name, fn in RETRIEVERS.items():
        ids = retrieved_ids(fn(q_text))
        per_query[name]["ndcg"].append(ndcg_at_k(ids, q_rels, k=10))
        per_query[name]["recall"].append(recall_at_k(ids, q_rels, k=10))

full_rows = []
for name in RETRIEVERS:
    ndcg_mean, ndcg_lo, ndcg_hi = bootstrap_ci(per_query[name]["ndcg"], n_bootstrap=1000)
    rec_mean,  rec_lo,  rec_hi  = bootstrap_ci(per_query[name]["recall"], n_bootstrap=1000)
    full_rows.append({
        "approach":   name,
        "nDCG@10":    ndcg_mean,
        "nDCG@10_lo": ndcg_lo, "nDCG@10_hi": ndcg_hi,
        "Recall@10":    rec_mean,
        "Recall@10_lo": rec_lo, "Recall@10_hi": rec_hi,
    })

display(Markdown(f"**Live 2K eval complete** — {len(full_2k)} queries, {len(RETRIEVERS)} retrievers, bootstrap CIs from 1,000 resamples."))


In [ ]:
full_df = pd.DataFrame(full_rows)


def fmt_ci(row, base):
    return f"{row[base]:.3f}  [{row[base + '_lo']:.3f}, {row[base + '_hi']:.3f}]"


full_df["nDCG@10 (95% CI)"]            = full_df.apply(lambda r: fmt_ci(r, "nDCG@10"),   axis=1)
full_df["Recall@10 (95% CI)"]          = full_df.apply(lambda r: fmt_ci(r, "Recall@10"), axis=1)
full_df["nDCG@10 (plain English)"]     = full_df.apply(lambda r: explain_metric("nDCG@10",   r["nDCG@10"]),   axis=1)
full_df["Recall@10 (plain English)"]   = full_df.apply(lambda r: explain_metric("Recall@10", r["Recall@10"]), axis=1)

display(Markdown("#### Full 2K ESCI test set — *the headline claim*"))
display(full_df[[
    "approach",
    "nDCG@10 (95% CI)", "nDCG@10 (plain English)",
    "Recall@10 (95% CI)", "Recall@10 (plain English)",
]].style.hide(axis="index"))


### Bad queries: when routing matters

The model was fine-tuned to map any input to product titles in this catalog. So when you give it a general-knowledge query — `year world war 2 ended` — it still returns products. That's not the model breaking; it's the model doing exactly what it was trained to do.

**Intent detection belongs upstream.** In production you route shopping vs non-shopping queries before they hit retrieval. Don't expect a fine-tuned retrieval model to recognize "this isn't a shopping query and I should abstain" — that's a different system.

Five bad queries below. Different intents, all routed to products.

In [ ]:
for i, q in enumerate(bad_queries):
    results = search_splade(q["text"])
    render_query_block(q["text"], results, expanded=(i == 0))


### Stretch — your turn

If we have time, type a query you actually care about. We'll fire it through all five retrievers and look at the results together.

In [ ]:
your_query = "ergonomic standing desk converter dual monitor"  # <-- edit me

results = {name: fn(your_query) for name, fn in RETRIEVERS.items()}
compare_results(query=your_query, models_results=results, esci_qrels=None)


### Q&A

Open floor. A few prompts to seed it:

- **"What about cross-encoder rerankers?"** — That's the next layer of the production stack. Hybrid retrieval → cross-encoder rerank on the top-100 → final top-10. Deferred to a follow-up workshop.
- **"How big does my training set need to be?"** — The model you used today was trained on ~100K query-product pairs in ~6 min on an A100. Diminishing returns kick in well before 1M.
- **"Can I just keep using BM25?"** — Look at the gap on the wrap table. BM25 is a hard floor, not a ceiling.
- **"What if my queries shift over time?"** — Schedule re-tuning. The fine-tuning cost is low. Catastrophic forgetting risk is real if you cross domains — see the self-study notebook.

---
### Takeaway links

- **Article series — Sparse Embeddings for E-commerce (part 1)** — https://qdrant.tech/articles/sparse-embeddings-ecommerce-part-1/
- **Fine-tuned model on HuggingFace** — https://huggingface.co/thierrydamiba/splade-ecommerce-esci
- **Collection setup (replicate elsewhere)** — `scripts/setup_collections.py`
- **Training notebook (self-study)** — `notebooks/splade_training.ipynb` in this repo
- **Amazon ESCI dataset** — https://github.com/amazon-science/esci-data

Thanks for coming. Go fine-tune something.